# Phase III — Scale anatomy, resolution robustness, artist geometry, and *The Starry Night*

This notebook starts after the successful Phase I and Phase II experiments. Its goal is **interpretation**, not merely higher classification performance.

Scientific questions:
1. Which Gaussian scales carry the artist-discriminative curvature signal?
2. Are geometric descriptors stable when the same painting is resized?
3. Which descriptors differ most strongly among artists?
4. Where does *The Starry Night* lie within Van Gogh's empirical geometry?

For resolution robustness we use the Phase-III derivative-of-Gaussian implementation and the dimensionless quantity

$$\tilde\kappa_\sigma = \sigma_{\rm px}\,\kappa_\sigma,$$

with smoothing scales matched relatively across 256, 512, and 1024 px. Phase-I features remain unchanged for the scale-ablation and corpus-position analyses so the experimental chain stays comparable.


In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

# Important when rerunning in Colab: leave the repository before deleting it.
os.chdir("/content")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")
], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repository:", REPO_DIR)
print("Branch:", BRANCH)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


## 1. Recover Phase-I and Phase-II outputs

Upload both ZIPs when prompted:
- `painting_geometry_first_results.zip`
- `painting_geometry_phase2_results.zip`

Only the required CSVs are extracted. The 512 px Phase-I curvature features are reused.


In [ ]:
from google.colab import files
import io, zipfile, pandas as pd, numpy as np

INPUT_DIR = Path("/content/phase3_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED = {"features_train_multiscale.csv", "features_test_multiscale.csv"}
OPTIONAL = {"excluded_test_images.csv"}

def extract_needed(blob: bytes):
    found = set()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for member in z.namelist():
            base = Path(member).name
            if base in REQUIRED | OPTIONAL:
                with z.open(member) as src, open(INPUT_DIR / base, "wb") as dst:
                    dst.write(src.read())
                found.add(base)
    return found

if not all((INPUT_DIR / x).exists() for x in REQUIRED):
    print("Upload the Phase-I and Phase-II ZIP files.")
    uploaded = files.upload()
    found = set()
    for name, blob in uploaded.items():
        if name.lower().endswith(".zip"):
            found |= extract_needed(blob)
        elif Path(name).name in REQUIRED | OPTIONAL:
            (INPUT_DIR / Path(name).name).write_bytes(blob)
            found.add(Path(name).name)
    print("Recovered:", sorted(found))

missing = [x for x in REQUIRED if not (INPUT_DIR / x).exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

TRAIN_CSV = INPUT_DIR / "features_train_multiscale.csv"
TEST_CSV = INPUT_DIR / "features_test_multiscale.csv"
EXCLUDED_TEST = INPUT_DIR / "excluded_test_images.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
print("Train:", train_df.shape, "Test:", test_df.shape)
print("Curvature columns:", sum(c.startswith("curv__") for c in train_df.columns))
print("Orientation columns:", sum(c.startswith("orient__") for c in train_df.columns))
print("Clean exclusion file present:", EXCLUDED_TEST.exists())


## 2. Scale ablation at 512 px

Each single-scale model uses the same number of curvature descriptors. Hyperparameter selection is performed only inside the training split. The best single scale is chosen by training CV, then compared with multiscale combinations.


In [ ]:
PHASE3_SCALE = REPO_DIR / "results" / "phase3_scale"
PHASE3_SCALE.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "scripts/run_scale_ablation.py",
    "--train", str(TRAIN_CSV),
    "--test", str(TEST_CSV),
    "--output-dir", str(PHASE3_SCALE),
    "--cv-folds", "3",
    "--n-jobs", "-1",
]
subprocess.run(cmd, check=True)

scale_results = pd.read_csv(PHASE3_SCALE / "scale_ablation_results.csv")
scale_deltas = pd.read_csv(PHASE3_SCALE / "scale_ablation_deltas.csv")
display(scale_results)
display(scale_deltas)


In [ ]:
import matplotlib.pyplot as plt
x = np.arange(len(scale_results))
plt.figure(figsize=(11, 4.5))
plt.errorbar(
    x, scale_results["macro_f1"],
    yerr=[scale_results["macro_f1"]-scale_results["macro_f1_ci_low"],
          scale_results["macro_f1_ci_high"]-scale_results["macro_f1"]],
    fmt="o", capsize=3
)
plt.xticks(x, scale_results["experiment"], rotation=45, ha="right")
plt.ylabel("Macro-F1")
plt.title("Phase III — scale-specific geometry ablation")
plt.tight_layout()
plt.savefig(PHASE3_SCALE / "Figure_scale_ablation_macroF1.png", dpi=220, bbox_inches="tight")
plt.show()


## 3. Artist-wise geometry and effect sizes

For every curvature/orientation descriptor we compute Kruskal-Wallis, Benjamini-Hochberg FDR, and an effect size. The strongest descriptors are then examined with pairwise Mann-Whitney tests and rank-biserial effects.


In [ ]:
PHASE3_ARTIST = REPO_DIR / "results" / "phase3_artist"
PHASE3_ARTIST.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "scripts/analyze_artist_geometry.py",
    "--train", str(TRAIN_CSV),
    "--test", str(TEST_CSV),
    "--output-dir", str(PHASE3_ARTIST),
    "--top-n", "10",
]
if EXCLUDED_TEST.exists():
    cmd += ["--excluded-test", str(EXCLUDED_TEST)]
subprocess.run(cmd, check=True)

artist_global = pd.read_csv(PHASE3_ARTIST / "artist_geometry_global.csv")
artist_pairwise = pd.read_csv(PHASE3_ARTIST / "artist_geometry_pairwise.csv")
artist_summary = pd.read_csv(PHASE3_ARTIST / "artist_geometry_summary_by_artist.csv")
display(artist_global.head(15))


In [ ]:
top_features = artist_global.head(8)["feature"].tolist()
m = artist_summary[artist_summary["feature"].isin(top_features)].pivot(
    index="artist", columns="feature", values="median"
)
med = m.median(axis=0)
iqr = m.quantile(0.75, axis=0) - m.quantile(0.25, axis=0)
z = (m - med) / iqr.replace(0, np.nan)

plt.figure(figsize=(12, 5.5))
im = plt.imshow(z.to_numpy(), aspect="auto")
plt.colorbar(im, label="Median profile / cross-artist IQR")
plt.yticks(np.arange(len(z.index)), z.index)
plt.xticks(np.arange(len(z.columns)), [c.replace("curv__", "") for c in z.columns], rotation=70, ha="right")
plt.title("Artist geometry profiles — top effect-size descriptors")
plt.tight_layout()
plt.savefig(PHASE3_ARTIST / "Figure_artist_geometry_profiles.png", dpi=220, bbox_inches="tight")
plt.show()


## 4. Resolution robustness

A balanced subset is recomputed at 256, 512, and 1024 px. Reference scales are defined at 512 px and transformed as

$$\sigma_{\rm px}(R)=\sigma_{\rm ref}\frac{R}{512}.$$

The dimensionless curvature $\tilde\kappa=\sigma_{\rm px}\kappa$ is then compared across resolutions using Spearman correlation, ICC(3,1), and robust drift.


In [ ]:
import kagglehub
DATASET_DIR = Path(kagglehub.dataset_download("delayedkarma/impressionist-classifier-data"))
print("Dataset:", DATASET_DIR)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

def artist_root_score(path: Path):
    n_artists, n_images = 0, 0
    for d in [p for p in path.iterdir() if p.is_dir()]:
        imgs = [p for p in d.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
        if imgs:
            n_artists += 1
            n_images += len(imgs)
    return n_artists, n_images

def find_split_root(name: str):
    candidates = [p for p in DATASET_DIR.rglob(name) if p.is_dir()]
    scored = [(artist_root_score(p), p) for p in candidates]
    scored = [x for x in scored if x[0][0] >= 2]
    if not scored:
        raise FileNotFoundError(f"Could not resolve artist root for split={name}")
    scored.sort(key=lambda x: (x[0][0], x[0][1]), reverse=True)
    return scored[0][1]

TRAIN_ROOT = find_split_root("training")
print("Training root:", TRAIN_ROOT, artist_root_score(TRAIN_ROOT))


In [ ]:
PHASE3_RES = REPO_DIR / "results" / "phase3_resolution"
PHASE3_RES.mkdir(parents=True, exist_ok=True)
ROBUSTNESS_PER_ARTIST = 30

cmd = [
    sys.executable, "-u", "scripts/run_resolution_robustness.py",
    "--root", str(TRAIN_ROOT),
    "--output-dir", str(PHASE3_RES),
    "--per-artist", str(ROBUSTNESS_PER_ARTIST),
    "--seed", "42",
    "--resolutions", "256", "512", "1024",
    "--sigma-refs", "1", "2", "4", "8",
    "--reference-long-side", "512",
]
subprocess.run(cmd, check=True)

robustness = pd.read_csv(PHASE3_RES / "resolution_robustness_summary.csv")
rho_cols = ["spearman_256_512", "spearman_512_1024", "spearman_256_1024"]
robustness["min_pairwise_rho"] = robustness[rho_cols].min(axis=1)
stable = robustness[(robustness["icc3_1"] >= 0.75) & (robustness["min_pairwise_rho"] >= 0.80)].copy()
print(f"Stable descriptors: {len(stable)} / {len(robustness)}")
display(robustness.sort_values("icc3_1", ascending=False).head(25))

plt.figure(figsize=(7, 5))
plt.scatter(robustness["min_pairwise_rho"], robustness["icc3_1"], alpha=0.7)
plt.axvline(0.80, linestyle="--")
plt.axhline(0.75, linestyle="--")
plt.xlabel("Minimum pairwise Spearman rho")
plt.ylabel("ICC(3,1)")
plt.title("Resolution stability of geometry descriptors")
plt.tight_layout()
plt.savefig(PHASE3_RES / "Figure_resolution_stability.png", dpi=220, bbox_inches="tight")
plt.show()


## 5. Position *The Starry Night* within Van Gogh

Upload one reliable JPG/PNG/TIF reproduction when prompted. The image is processed at 512 px with the same Phase-I geometry used for the corpus benchmark. Outputs are descriptive: robust multivariate distance, within-Van-Gogh percentile, nearest neighbors, feature percentiles, and PCA coordinates.


In [ ]:
PHASE3_STARRY = REPO_DIR / "results" / "phase3_starry"
PHASE3_STARRY.mkdir(parents=True, exist_ok=True)

print("Upload one image of The Starry Night.")
uploaded = files.upload()
image_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
image_names = [name for name in uploaded if Path(name).suffix.lower() in image_exts]
if not image_names:
    raise FileNotFoundError("No supported image uploaded.")

chosen = image_names[0]
STARRY_PATH = Path("/content") / Path(chosen).name
STARRY_PATH.write_bytes(uploaded[chosen])
print("Using:", STARRY_PATH)

cmd = [
    sys.executable, "scripts/position_starry_night.py",
    "--image", str(STARRY_PATH),
    "--train", str(TRAIN_CSV),
    "--test", str(TEST_CSV),
    "--output-dir", str(PHASE3_STARRY),
    "--artist", "VanGogh",
    "--long-side", "512",
]
if EXCLUDED_TEST.exists():
    cmd += ["--excluded-test", str(EXCLUDED_TEST)]
subprocess.run(cmd, check=True)

star_summary = pd.read_csv(PHASE3_STARRY / "starry_night_position_summary.csv")
star_features = pd.read_csv(PHASE3_STARRY / "starry_night_feature_percentiles.csv")
display(star_summary)
display(star_features.head(15))


In [ ]:
pca_df = pd.read_csv(PHASE3_STARRY / "starry_night_pca_coordinates.csv")
ref = pca_df[pca_df["kind"] == "reference"]
star = pca_df[pca_df["kind"] == "starry_night"]
plt.figure(figsize=(7, 6))
plt.scatter(ref["PC1"], ref["PC2"], alpha=0.35, label="Van Gogh reference")
plt.scatter(star["PC1"], star["PC2"], marker="*", s=220, label="The Starry Night")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("The Starry Night within Van Gogh geometry space")
plt.legend()
plt.tight_layout()
plt.savefig(PHASE3_STARRY / "Figure_starry_night_within_vangogh.png", dpi=220, bbox_inches="tight")
plt.show()


## 6. Compact readout and package outputs

Interpret scale results using training-CV selection; resolution stability using both rank preservation and ICC; artist contrasts using effect sizes plus FDR; and *The Starry Night* only as a corpus-position result, not an emotion, intention, or authenticity score.


In [ ]:
best_single = scale_results[scale_results["experiment"].isin(["S1", "S2", "S4", "S8"])].sort_values("best_cv_macro_f1", ascending=False).iloc[0]
print("Best single scale by TRAINING CV:", best_single["experiment"])
print("Validation Macro-F1:", round(best_single["macro_f1"], 4))
print("Top artist-level effect-size feature:", artist_global.iloc[0][["feature", "epsilon_squared", "q_fdr_bh"]].to_dict())
print("Resolution-stable descriptors:", len(stable), "/", len(robustness))
print("Starry Night distance percentile:", float(star_summary.loc[0, "distance_percentile_within_artist"]))


In [ ]:
import json
metadata = {
    "branch": BRANCH,
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "phase1_train_shape": list(train_df.shape),
    "phase1_test_shape": list(test_df.shape),
    "robustness_per_artist": ROBUSTNESS_PER_ARTIST,
    "resolutions": [256, 512, 1024],
    "sigma_refs_at_512": [1, 2, 4, 8],
}
(REPO_DIR / "results" / "phase3_run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

PACKAGE_DIR = Path("/content/painting_geometry_phase3_results")
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir()
for folder_name in ["phase3_scale", "phase3_artist", "phase3_resolution", "phase3_starry"]:
    src = REPO_DIR / "results" / folder_name
    if src.exists():
        shutil.copytree(src, PACKAGE_DIR / folder_name)
shutil.copy2(REPO_DIR / "results" / "phase3_run_metadata.json", PACKAGE_DIR / "phase3_run_metadata.json")

zip_path = shutil.make_archive("/content/painting_geometry_phase3_results", "zip", root_dir=PACKAGE_DIR)
print("Created:", zip_path)
files.download(zip_path)
